# 01 — Exploratory Data Analysis
**Project:** Behavioral Anomaly Detection in Human Chess  
**Course:** BCSAI - Machine Learning Foundations  

This notebook performs exploratory data analysis on the Lichess games dataset.  
Objectives:
- Understand the dataset structure and basic statistics
- Identify data quality issues (missing values, outliers)
- Visualize rating distributions, time controls, and game outcomes
- Motivate our feature engineering choices

In [ ]:
from pathlib import Path
import sys

_root = Path.cwd().resolve()
if not (_root / "src").is_dir():
    _root = (_root / "..").resolve()
sys.path.insert(0, str(_root))

RESULTS_DIR = _root / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_palette("muted")

from src.data_loader import load_raw, validate_schema, parse_time_control, clean, to_player_level

## 1. Load Raw Data

In [ ]:
df = load_raw()  # loads from data/raw/games.csv
print(f'Shape: {df.shape}')
df.head()

In [ ]:
validate_schema(df)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nMissing values per column:')
print(df.isnull().sum())

## 2. Rating Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['white_rating'], bins=50, alpha=0.6, label='White', color='steelblue')
axes[0].hist(df['black_rating'], bins=50, alpha=0.6, label='Black', color='coral')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Rating Distribution by Color')
axes[0].legend()

rating_diff = df['white_rating'] - df['black_rating']
axes[1].hist(rating_diff, bins=50, color='mediumseagreen')
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Rating Differential (White - Black)')
axes[1].set_title('Rating Differential Distribution')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'eda_ratings.png'), dpi=150)
plt.show()

print(f'Rating stats (white):\n{df["white_rating"].describe()}')

## 3. Time Controls

In [ ]:
df_tc = parse_time_control(df)
tc_counts = df_tc['time_control_cat'].value_counts()
print('Time control distribution:')
print(tc_counts)

tc_counts.plot(kind='bar', color='steelblue', rot=0)
plt.title('Games by Time Control')
plt.ylabel('Number of Games')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'eda_time_controls.png'), dpi=150)
plt.show()

## 4. Game Outcomes

In [ ]:
outcome = df['winner'].fillna('draw').value_counts()
print('Game outcomes:', outcome.to_dict())

fig, ax = plt.subplots(figsize=(6, 4))
outcome.plot(kind='bar', ax=ax, color=['steelblue', 'coral', 'gray'], rot=0)
ax.set_title('Game Outcomes')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'eda_outcomes.png'), dpi=150)
plt.show()

## 5. Opening Depth vs Rating

In [ ]:
df_clean = clean(df_tc)

# Sample for scatter plot
sample = df_clean.sample(min(3000, len(df_clean)), random_state=42)
avg_rating = (sample['white_rating'] + sample['black_rating']) / 2

plt.scatter(avg_rating, sample['opening_ply'], alpha=0.3, s=5, color='steelblue')
plt.xlabel('Average Rating')
plt.ylabel('Opening Ply (Theory Depth)')
plt.title('Opening Sophistication vs Rating')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'eda_opening_ply.png'), dpi=150)
plt.show()

corr = avg_rating.corr(sample['opening_ply'])
print(f'Correlation between average rating and opening depth: {corr:.3f}')

## 6. Player-Level Summary

In [ ]:
player_df = to_player_level(df_clean)
games_per_player = player_df.groupby('player_id').size()

print(f'Total players: {player_df["player_id"].nunique():,}')
print(f'Games per player: {games_per_player.describe()}')

games_per_player.clip(upper=50).hist(bins=30, color='steelblue')
plt.xlabel('Number of Games')
plt.ylabel('Number of Players')
plt.title('Games per Player Distribution (clipped at 50)')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'eda_games_per_player.png'), dpi=150)
plt.show()

## 7. Key EDA Takeaways

*(Fill in after running the notebook — summarize what you found)*

- Rating distribution: ...
- Time control breakdown: ...
- Missing data: ...
- Opening depth correlation with rating: ...
- Any outliers or quality issues: ...